# 02 — Three Encoders, Three Masks, One Training Loop

**By the end of this notebook** you'll have run a real PRAGMA pretraining loop, watched the MLM loss fall over 40 steps, saved a checkpoint, reloaded it, and verified the model produces bit-identical output.

## What this notebook teaches

- The three-encoder architecture: Profile State Encoder → Event Encoder → History Encoder (§2.3)
- Why three encoders? Each sees a different granularity of the customer's history
- The three masking strategies: token masking, event masking, key-type masking (§2.3.5)
- What a masked event modelling (MEM) training step looks like in code
- How to save and reload a checkpoint with a bit-identical-output verification

## Prerequisites

Run notebook 01 first — this notebook imports the same tokenisation concepts without re-explaining them.

**How to use:** run every cell in order with Shift+Enter.

In [ ]:
# ── Setup: repo root on sys.path ─────────────────────────────────────────────
import sys, pathlib, tempfile

# Locate repo root by searching candidate paths for pyproject.toml.
# Works whether Jupyter CWD is:
#   - the repo root itself           (local dev, top-level launch)
#   - notebooks/                     (local dev, launched from subdir)
#   - /opt/app-root/src              (OpenShift AI Workbench — CWD is parent of repo)
#   - /opt/app-root/src/pragma-encoder  (workbench with repo as CWD)
_here = pathlib.Path().resolve()
repo_root = next(
    (p for p in [_here, _here.parent,
                 _here / "pragma-encoder", _here.parent / "pragma-encoder"]
     if (p / "pyproject.toml").exists()),
    _here,
)
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

# ── Standard library ──────────────────────────────────────────────────────────
import math

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch

# ── PRAGMA model (src/pragma_encoder/model/) ──────────────────────────────────
from pragma_encoder.model        import PRAGMA, PRAGMAConfig          # full model + config
from pragma_encoder.model.assembler import EmbeddingAssembler          # IDs → embedding tensors

# ── PRAGMA masking (src/pragma_encoder/masking/) ──────────────────────────────
from pragma_encoder.masking      import MaskingStrategy                # three-strategy masker

# ── PRAGMA vocabulary (src/pragma_encoder/tokenizer/) ────────────────────────
from pragma_encoder.tokenizer.vocabulary import VocabularySpec         # token ID ranges

matplotlib.rcParams['figure.dpi'] = 110
torch.manual_seed(42)
print(f"Setup complete. repo_root={repo_root}")
print("PyTorch", torch.__version__)

## The three-encoder architecture

PRAGMA stacks three specialised Transformer encoders, each processing a different granularity of a customer's history (§2.3):

| Encoder | Input | Output | Paper section | Source file |
|---|---|---|---|---|
| **Profile State Encoder** | Profile attribute fields (age, account type, …) | `[USR]` token — a single vector summarising the customer | §2.3.2 | `src/pragma_encoder/encoders/profile_state_encoder.py` |
| **Event Encoder** | One transaction's field tokens `(key, value)` | `[EVT]` token — a single vector summarising one event | §2.3.3 | `src/pragma_encoder/encoders/event_encoder.py` |
| **History Encoder** | `[USR : EVT₁ : EVT₂ : … : EVTₙ]` concatenated | Contextualised `[EVT]` vectors, conditioned on profile | §2.3.4 | `src/pragma_encoder/encoders/history_encoder.py` |

The key insight is the concatenation in the History Encoder (Equation 6). By placing `[USR]` at position 0 and letting every `[EVT]` attend to it through standard bidirectional self-attention, the profile conditions every event representation — without any cross-attention sublayer.

All three encoders use **full bidirectional attention** — PRAGMA is an encoder-only model, never causal.

In [ ]:
# ── Initialise PRAGMA-S ───────────────────────────────────────────────────────
# PRAGMA-S is the smallest scale: ~10 M parameters, d_model=192 (Table 1, §2.3.1)
config = PRAGMAConfig.pragma_s()

n_total = sum(p.numel() for p in PRAGMA(config).parameters())
print(f"PRAGMA-S  d_model={config.d_model}  n_heads={config.n_heads}")
print(f"  profile_encoder_layers = {config.profile_encoder_layers}")
print(f"  event_encoder_layers   = {config.event_encoder_layers}")
print(f"  history_encoder_layers = {config.history_encoder_layers}")
print(f"  ~{n_total/1e6:.1f} M parameters total")

## Build a synthetic vocabulary and batch

The model works with integer token IDs. We need a `VocabularySpec` that tells `EmbeddingAssembler` where key IDs end and value IDs begin, and where special tokens live. For this notebook we construct a minimal synthetic spec — no real transaction data required.

In [ ]:
# ── Vocabulary ────────────────────────────────────────────────────────────────
# Special tokens: PAD=0, MASK=1, EVT=2, SEP=3  (four reserved IDs)
# Key IDs  : [4, 4 + key_vocab_size)
# Value IDs: [4 + key_vocab_size, 4 + key_vocab_size + value_vocab_size)
vocab_spec = VocabularySpec(
    special_tokens={"PAD": 0, "MASK": 1, "EVT": 2, "SEP": 3},
    key_start=4,
    key_size=config.key_vocab_size,
    value_start=4 + config.key_vocab_size,
    value_size=config.value_vocab_size,
    total_embedding_vocab_size=4 + config.key_vocab_size + config.value_vocab_size,
    field_key_ids={},
    field_value_ranges={},
)
print(f"Vocabulary: {vocab_spec.total_embedding_vocab_size:,} tokens total")
print(f"  Key range  : [{vocab_spec.key_start}, {vocab_spec.key_start + vocab_spec.key_size})")
print(f"  Value range: [{vocab_spec.value_start}, {vocab_spec.value_start + vocab_spec.value_size})")

# ── Batch dimensions ─────────────────────────────────────────────────────────
# B  = batch size (customers)
# NE = number of events (transactions) per customer
# NI = number of value tokens per event (field slots)
# NA = number of profile attribute tokens
B, NE, NI, NA = 4, 10, 8, 6
print(f"\nSynthetic batch: B={B}, NE={NE} events, NI={NI} fields/event, NA={NA} profile attrs")

In [ ]:
# ── Synthetic random batch ────────────────────────────────────────────────────
# Key IDs are in the key range; value IDs are in the value range.
# Times are log-seconds coordinates (small positive floats).
rng = torch.Generator()
rng.manual_seed(0)

def _rand_keys(size):
    return torch.randint(vocab_spec.key_start,
                         vocab_spec.key_start + vocab_spec.key_size,
                         size, generator=rng)

def _rand_vals(size):
    return torch.randint(vocab_spec.value_start,
                         vocab_spec.value_start + vocab_spec.value_size,
                         size, generator=rng)

# Within-field position IDs: 0, 1, 2, ... NI-1 (broadcast across batch/events)
xa_pos = torch.arange(NA).unsqueeze(0).expand(B, -1)        # (B, NA)
xe_pos = torch.arange(NI).unsqueeze(0).unsqueeze(0).expand(B, NE, -1)  # (B, NE, NI)

# te has shape (B, 1+NE): position 0 is the [USR] token, positions 1..NE are events
te_base = torch.rand(B, NE, generator=rng) * 80.0
te_usr  = torch.zeros(B, 1)                                  # [USR] at time 0
te_full = torch.cat([te_usr, te_base], dim=1)                # (B, 1+NE)

batch = {
    # Profile attribute key / value / position tokens  (B, NA)
    "xa_key_ids":   _rand_keys((B, NA)),
    "xa_val_ids":   _rand_vals((B, NA)),
    "xa_pos_ids":   xa_pos,
    "ta":           torch.rand(B, NA, generator=rng) * 5.0,  # log-seconds times

    # Event key / value / position tokens              (B, NE, NI)
    "xe_key_ids":   _rand_keys((B, NE, NI)),
    "xe_val_ids":   _rand_vals((B, NE, NI)),
    "xe_pos_ids":   xe_pos,

    # Calendar features [hour, day_of_week, day_of_month]  (B, NE, 3)
    "xt":           torch.rand(B, NE, 3, generator=rng),

    # History temporal coordinates (B, 1+NE): [USR] slot + one per event
    "te":           te_full,
}
print("Batch tensor shapes:")
for k, v in batch.items():
    print(f"  {k:20s}: {tuple(v.shape)}")

## The three masking strategies

PRAGMA uses **masked event modelling** (MEM) — a BERT-style objective applied to transaction sequences (§2.3.5). Three masking strategies are ORed together on each batch:

| Strategy | What gets masked | Probability |
|---|---|---|
| **Token masking** | Individual value tokens within events | 15% of tokens |
| **Event masking** | All tokens in a whole event | 10% of events |
| **Key-type masking** | All tokens sharing the same field key across events | 10% of key types |

Of the selected positions: 90% → `[MASK]` token (supervised — included in loss), 10% → random `[UNK]` replacement (excluded from loss). This is the same 80/10/10 split used in BERT, adapted for structured event sequences.

Source: `src/pragma_encoder/masking/strategy.py::MaskingStrategy`

In [ ]:
masker = MaskingStrategy(config)

# Apply masking to a single batch
# Returns: (masked_val_ids, original_val_ids, supervision_mask)
# supervision_mask is True where the model must predict the original token
masked_val_ids, original_val_ids, mlm_mask = masker.forward(
    batch["xe_val_ids"],   # (B, NE, NI)
    batch["xe_key_ids"],   # (B, NE, NI)
)

# Guard: if no positions were selected (can happen with very small batches),
# force at least one supervised position so loss is defined
if not mlm_mask.any():
    mlm_mask[0, 0, 0]       = True
    masked_val_ids[0, 0, 0] = vocab_spec.special_tokens["MASK"]

n_total_positions = mlm_mask.numel()
n_masked          = mlm_mask.sum().item()
print(f"Total positions : {n_total_positions}")
print(f"Masked positions: {n_masked}  ({100 * n_masked / n_total_positions:.1f}%)")
print(f"masked_val_ids shape : {tuple(masked_val_ids.shape)}")
print(f"mlm_mask shape       : {tuple(mlm_mask.shape)}")
print()

# Peek at one event: original vs masked
ev = 0
print("Event 0, customer 0 — first 4 field slots:")
print(f"  original : {batch['xe_val_ids'][0, ev, :4].tolist()}")
print(f"  masked   : {masked_val_ids[0, ev, :4].tolist()}")
print(f"  mask     : {mlm_mask[0, ev, :4].tolist()}")

## Set up model and optimiser

In [ ]:
model     = PRAGMA(config)
assembler = EmbeddingAssembler(vocab_spec, config)

# Adam optimiser over both model and assembler parameters
optimizer = torch.optim.Adam(
    list(model.parameters()) + list(assembler.parameters()),
    lr=1e-4,
)

n_model     = sum(p.numel() for p in model.parameters())
n_assembler = sum(p.numel() for p in assembler.parameters())
print(f"Model parameters     : {n_model:,}")
print(f"Assembler parameters : {n_assembler:,}   (embedding lookup table)")
print(f"Total trainable      : {n_model + n_assembler:,}")

## One training step — what happens inside

A single MEM step:

1. **Mask** — `MaskingStrategy` selects positions to hide and replaces them with `[MASK]`
2. **Assemble** — `EmbeddingAssembler` converts integer IDs to float embedding tensors for all three encoder inputs
3. **Forward** — `PRAGMA` runs Profile State Encoder → Event Encoder → History Encoder in order
4. **Loss** — `model.mlm_head.compute_loss(logits, targets)` computes cross-entropy over the masked positions only
5. **Backward + step** — standard gradient descent

In [ ]:
def training_step(batch: dict) -> float:
    """One masked event modelling training step. Returns scalar loss."""
    optimizer.zero_grad()

    # Step 1 — mask
    masked_val_ids, _, mlm_mask = masker.forward(
        batch["xe_val_ids"], batch["xe_key_ids"]
    )
    if not mlm_mask.any():
        mlm_mask[0, 0, 0]       = True
        masked_val_ids[0, 0, 0] = vocab_spec.special_tokens["MASK"]

    # Step 2 — assemble IDs into embedding tensors
    assembled = assembler.forward(
        xa_key_ids=batch["xa_key_ids"],
        xa_val_ids=batch["xa_val_ids"],
        xa_pos_ids=batch["xa_pos_ids"],
        ta=batch["ta"],
        xe_key_ids=batch["xe_key_ids"],
        xe_val_ids=masked_val_ids,        # masked value tokens
        xe_pos_ids=batch["xe_pos_ids"],
        xt=batch["xt"],                   # calendar features
        te=batch["te"],
        target_ids=batch["xe_val_ids"],   # original (unmasked) targets
        mask=mlm_mask,
    )

    # Step 3 — forward through all three encoders
    output = model.forward(
        xa=assembled.xa,
        ta=assembled.ta,
        xe=assembled.xe,
        xt=assembled.xt,
        te=assembled.te,
        mask=assembled.mlm_mask,
    )

    # Step 4 — loss at masked positions only
    logits        = output["logits"]
    valid_targets = assembled.targets[assembled.mlm_mask]
    loss = model.mlm_head.compute_loss(logits, valid_targets)

    # Step 5 — backward
    loss.backward()
    optimizer.step()
    return loss.item()

# Verify one step runs without error
loss_0 = training_step(batch)
print(f"Step 0 loss: {loss_0:.4f}  ✓")

## Short training loop — 40 steps

In [ ]:
N_STEPS = 40
losses  = []

# Re-initialise so the loss curve starts fresh from step 0
torch.manual_seed(42)
model     = PRAGMA(config)
assembler = EmbeddingAssembler(vocab_spec, config)
optimizer = torch.optim.Adam(
    list(model.parameters()) + list(assembler.parameters()), lr=1e-4
)

print(f"Training PRAGMA-S for {N_STEPS} steps on synthetic data...")
for step in range(N_STEPS):
    l = training_step(batch)
    losses.append(l)
    if step == 0 or (step + 1) % 10 == 0:
        print(f"  step {step+1:3d}  loss={l:.4f}")

print(f"\nFirst loss : {losses[0]:.4f}")
print(f"Last loss  : {losses[-1]:.4f}")
print(f"Reduction  : {losses[0] - losses[-1]:.4f}  ({'↓' if losses[-1] < losses[0] else '→'})")

## Visualisation — MLM loss curve

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(range(1, N_STEPS + 1), losses, color="#4C72B0", linewidth=2, label="MLM loss")

# Smoothed trend (rolling mean of 5)
if N_STEPS >= 5:
    smooth = np.convolve(losses, np.ones(5) / 5, mode="valid")
    ax.plot(range(3, N_STEPS - 1), smooth, color="#DD8452",
            linewidth=2, linestyle="--", label="5-step rolling mean")

ax.set_xlabel("Training step", fontsize=11)
ax.set_ylabel("MLM cross-entropy loss", fontsize=11)
ax.set_title("PRAGMA-S — masked event modelling loss on synthetic data (§2.3.5)", fontsize=12)
ax.legend(fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

**What you're looking at:** the blue line is the raw MLM loss at each step. The orange dashed line is a 5-step rolling mean — it smooths out the noise and makes the downward trend clearer. On 40 steps with a tiny synthetic batch, the loss will vary but should drift down overall. On real IBM TabFormer data with thousands of customers the trend is much smoother.

## Visualisation 2 — Masked input vs model prediction

Let's look at one event slot: what value was masked, and what did the model predict?

In [ ]:
model.eval()
with torch.no_grad():
    masked_val_ids_eval, _, mlm_mask_eval = masker.forward(
        batch["xe_val_ids"], batch["xe_key_ids"]
    )
    if not mlm_mask_eval.any():
        mlm_mask_eval[0, 0, 0]       = True
        masked_val_ids_eval[0, 0, 0] = vocab_spec.special_tokens["MASK"]

    assembled_eval = assembler.forward(
        xa_key_ids=batch["xa_key_ids"],
        xa_val_ids=batch["xa_val_ids"],
        xa_pos_ids=batch["xa_pos_ids"],
        ta=batch["ta"],
        xe_key_ids=batch["xe_key_ids"],
        xe_val_ids=masked_val_ids_eval,
        xe_pos_ids=batch["xe_pos_ids"],
        xt=batch["xt"],
        te=batch["te"],
        target_ids=batch["xe_val_ids"],
        mask=mlm_mask_eval,
    )
    out_eval = model.forward(
        xa=assembled_eval.xa, ta=assembled_eval.ta,
        xe=assembled_eval.xe, xt=assembled_eval.xt,
        te=assembled_eval.te, mask=assembled_eval.mlm_mask,
    )
    pred_ids = out_eval["logits"].argmax(dim=-1)   # shape: (n_masked_positions,)
model.train()

# Extract masked positions from customer 0 for a readable display
mask_c0   = mlm_mask_eval[0]                     # (NE, NI)
orig_c0   = batch["xe_val_ids"][0][mask_c0].tolist()
masked_c0 = masked_val_ids_eval[0][mask_c0].tolist()

# pred_ids are indexed over ALL masked positions across the batch;
# count how many belong to customers before customer 0
n_before = mlm_mask_eval[:0].sum().item()  # zero for customer 0
n_c0     = mask_c0.sum().item()
pred_c0  = pred_ids[int(n_before): int(n_before) + n_c0].tolist()

# Show first 8 masked positions
display_n = min(8, n_c0)
print(f"Customer 0 — {n_c0} masked positions (showing first {display_n}):")
print(f"{'Pos':>4}  {'Original':>10}  {'Masked':>10}  {'Predicted':>10}  Correct?")
for i in range(display_n):
    correct = "✓" if pred_c0[i] == orig_c0[i] else ""
    print(f"{i:>4}  {orig_c0[i]:>10}  {masked_c0[i]:>10}  {pred_c0[i]:>10}  {correct}")

In [ ]:
# Bar chart: how often does the model predict correctly for customer 0?
correct_count  = sum(p == o for p, o in zip(pred_c0, orig_c0))
incorrect_count = n_c0 - correct_count

fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(
    ["Correct", "Incorrect"],
    [correct_count, incorrect_count],
    color=["#55A868", "#C44E52"],
    width=0.5, edgecolor="white", linewidth=1.5,
)
for bar, val in zip(bars, [correct_count, incorrect_count]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            str(val), ha="center", va="bottom", fontsize=12, fontweight="bold")
ax.set_ylabel("Masked positions", fontsize=11)
ax.set_title(f"Model predictions after {N_STEPS} steps\n(customer 0, synthetic data)",
             fontsize=11)
ax.set_ylim(0, max(correct_count, incorrect_count) * 1.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

acc = correct_count / n_c0 if n_c0 > 0 else 0
print(f"\nMask prediction accuracy (customer 0): {acc:.1%}")
print("Note: 40 steps on random synthetic data is not enough to learn tokens.")
print("On real IBM TabFormer data with many more steps the model learns real patterns.")

## Checkpoint: save, reload, verify

In production, PRAGMA saves checkpoints to S3 (S3 checkpoint-resume is Level 5 in the OpenShift test suite). Here we save to a temporary file on disk and verify the reload produces bit-identical output — the same pattern used in `examples/workbench/06_local_learning_validation.py`.

In [ ]:
import tempfile, os

ckpt_dir  = pathlib.Path(tempfile.mkdtemp())
ckpt_path = ckpt_dir / "pragma-s-step40.pt"

# ── Save ─────────────────────────────────────────────────────────────────────
torch.save({
    "model_state_dict":     model.state_dict(),
    "assembler_state_dict": assembler.state_dict(),
    "config":               config,
    "step":                 N_STEPS,
    "final_loss":           losses[-1],
}, ckpt_path)
print(f"Checkpoint saved: {ckpt_path.name}  ({ckpt_path.stat().st_size / 1024:.0f} KB)")

# ── Reload into a fresh model instance ───────────────────────────────────────
ckpt            = torch.load(ckpt_path, weights_only=False)
config_reloaded = ckpt["config"]
model_reloaded  = PRAGMA(config_reloaded)
assembler_reloaded = EmbeddingAssembler(vocab_spec, config_reloaded)
model_reloaded.load_state_dict(ckpt["model_state_dict"])
assembler_reloaded.load_state_dict(ckpt["assembler_state_dict"])
print(f"Checkpoint reloaded (step={ckpt['step']}, loss={ckpt['final_loss']:.4f})")

# ── Bit-identical output check ────────────────────────────────────────────────
model.eval()
model_reloaded.eval()

with torch.no_grad():
    _no_mask = torch.zeros(B, NE, NI, dtype=torch.bool)
    assembled_check = assembler.forward(
        xa_key_ids=batch["xa_key_ids"], xa_val_ids=batch["xa_val_ids"],
        xa_pos_ids=batch["xa_pos_ids"], ta=batch["ta"],
        xe_key_ids=batch["xe_key_ids"], xe_val_ids=batch["xe_val_ids"],
        xe_pos_ids=batch["xe_pos_ids"], xt=batch["xt"], te=batch["te"],
        target_ids=batch["xe_val_ids"], mask=_no_mask,
    )
    out_orig = model.forward(
        xa=assembled_check.xa, ta=assembled_check.ta,
        xe=assembled_check.xe, xt=assembled_check.xt,
        te=assembled_check.te, mask=assembled_check.mlm_mask,
    )
    assembled_r = assembler_reloaded.forward(
        xa_key_ids=batch["xa_key_ids"], xa_val_ids=batch["xa_val_ids"],
        xa_pos_ids=batch["xa_pos_ids"], ta=batch["ta"],
        xe_key_ids=batch["xe_key_ids"], xe_val_ids=batch["xe_val_ids"],
        xe_pos_ids=batch["xe_pos_ids"], xt=batch["xt"], te=batch["te"],
        target_ids=batch["xe_val_ids"], mask=_no_mask,
    )
    out_reloaded = model_reloaded.forward(
        xa=assembled_r.xa, ta=assembled_r.ta,
        xe=assembled_r.xe, xt=assembled_r.xt,
        te=assembled_r.te, mask=assembled_r.mlm_mask,
    )

is_identical = torch.allclose(out_orig["zh"], out_reloaded["zh"])
print(f"\nBit-identical output check: {'PASS ✓' if is_identical else 'FAIL ✗'}")
if not is_identical:
    max_diff = (out_orig["zh"] - out_reloaded["zh"]).abs().max().item()
    print(f"  max absolute difference: {max_diff:.2e}  (should be 0)")

## What just happened — section recap

In this notebook you:

- Saw the three-encoder architecture: `ProfileStateEncoder` → `[USR]`, `EventEncoder` → `[EVT]`, `HistoryEncoder` → contextualised sequence
- Watched `MaskingStrategy` apply three strategies simultaneously (token, event, key-type masking)
- Ran 40 steps of masked event modelling on synthetic data and watched the loss decrease
- Saved a checkpoint and verified the reloaded model produces bit-identical output on the same input

The model you just trained is small and was trained on random data — it hasn't learned anything meaningful. But the training loop is identical to the one that runs on IBM TabFormer data in the OpenShift cluster (the Level 8 smoke test). The only difference is scale and data.

## Closing — what you now know

**Architecture:** PRAGMA is a three-encoder system. Profile State Encoder produces a `[USR]` token. Event Encoder produces one `[EVT]` token per transaction. History Encoder contextualises the full `[USR:EVT₁:…:EVTₙ]` sequence with bidirectional attention.

**Pretraining objective:** Masked event modelling — three masking strategies ORed together, loss computed only at masked positions. This is how the model learns to predict missing transaction fields from context.

**Checkpoint pattern:** `torch.save` / `torch.load(weights_only=False)` / `torch.allclose` for bit-identical verification.

**Next:** notebook 03 extracts the `[USR]` embeddings from the trained model and fits a linear probe (§3.1.1) for a synthetic binary classification task.

**Going deeper:** `docs/paper-to-code.md §2.3` maps every encoder to its source file. `examples/workbench/06_local_learning_validation.py` shows the same training loop with additional diagnostics. `docs/training-guide.md` covers real IBM TabFormer training.